# 99 — Cleanup (avoid ongoing AWS costs)

Run this notebook when you're done demoing.

It attempts to delete:
- SageMaker Endpoint (+ endpoint config + model)
- Model Monitor schedule
- CloudWatch dashboard
- Feature Store feature group (optional)

⚠️ Deletions can take a few minutes in AWS.


In [ ]:
import boto3
import sagemaker

In [ ]:
# Try to load saved variables (skip ones you don't have)
try:
    %store -r region
except Exception as e:
    region = boto3.Session().region_name

print("Region:", region)

for var in ["endpoint_name","schedule_name","dashboard_name","FEATURE_GROUP_NAME"]:
    try:
        get_ipython().run_line_magic("store", f"-r {var}")
        print(f"Loaded {var}: {globals().get(var)}")
    except Exception:
        print(f"{var} not found in %store (skipping)")

In [ ]:
sm = boto3.client("sagemaker", region_name=region)
cw = boto3.client("cloudwatch", region_name=region)

In [ ]:
# Delete monitoring schedule
if "schedule_name" in globals():
    try:
        sm.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print("Deleted monitoring schedule:", schedule_name)
    except Exception as e:
        print("Could not delete monitoring schedule:", e)

In [ ]:
# Delete endpoint (+ config + model)
if "endpoint_name" in globals():
    try:
        desc = sm.describe_endpoint(EndpointName=endpoint_name)
        endpoint_config_name = desc["EndpointConfigName"]
        print("EndpointConfigName:", endpoint_config_name)

        cfg = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
        model_names = [pv["ModelName"] for pv in cfg["ProductionVariants"]]
        print("Model(s):", model_names)

        sm.delete_endpoint(EndpointName=endpoint_name)
        print("Deleted endpoint:", endpoint_name)

        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print("Deleted endpoint config:", endpoint_config_name)

        for mn in model_names:
            try:
                sm.delete_model(ModelName=mn)
                print("Deleted model:", mn)
            except Exception as e:
                print("Could not delete model:", mn, e)

    except Exception as e:
        print("Could not delete endpoint resources:", e)

In [ ]:
# Delete CloudWatch dashboard
if "dashboard_name" in globals():
    try:
        cw.delete_dashboards(DashboardNames=[dashboard_name])
        print("Deleted dashboard:", dashboard_name)
    except Exception as e:
        print("Could not delete dashboard:", e)

In [ ]:
# Optional: delete feature group (careful — deletes feature store metadata)
if "FEATURE_GROUP_NAME" in globals():
    try:
        sm.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
        print("Deleted feature group:", FEATURE_GROUP_NAME)
    except Exception as e:
        print("Could not delete feature group:", e)